# Notebook 2 of 2 — Build the Silver layer

**Your job:** read your Bronze tables, find out what is wrong with the data, and publish **trustworthy Silver** tables, with the bad rows kept in a rejected table so nothing disappears silently.

> **Learning rule:** AI tools are allowed here. Understanding is not optional. You will explain every transformation in your own words in `OBSERVATION_REPORT.md`.

### The naming contract

| Output | ADLS path (container `lakehouse`) | Unity Catalog table |
|---|---|---|
| Clean trips | `silver/taxi_trips/` | `<catalog>.silver.taxi_trips` |
| Rejected trips | `silver/taxi_trips_rejected/` | `<catalog>.silver.taxi_trips_rejected` |
| Clean weather | `silver/weather_daily/` | `<catalog>.silver.weather_daily` |

External Delta tables again, same storage account and container as Bronze. Silver writes use `overwrite`, so the notebook can be re-run safely.

In [0]:
# ---- Fill in the two values marked <...> (same values as notebook 1) ----------
STORAGE_ACCOUNT = "dkdedls01"
CATALOG         = "sona_db_practices"

# ---- Do NOT change these ------------------------------------------------------
CONTAINER = "lakehouse"
BASE      = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
MONTHS    = ["2025-08", "2025-09", "2025-10"]

# The 20 columns exactly as the NYC TLC publishes them (Bronze keeps these names)
SRC_COLS = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance",
    "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID", "payment_type", "fare_amount",
    "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
    "congestion_surcharge", "Airport_fee", "cbd_congestion_fee",
]

---
## Step 1 — Look before you clean (profile Bronze)

Write queries that answer these four questions. Put the numbers in report §5.

1. How many rows in total, and how many nulls per column?
2. What is the earliest and latest pickup? How many pickups fall **outside the month of the file they came from**?
3. What values does `payment_type` take, and how many rows each?
4. How many trips have dropoff ≤ pickup, distance ≤ 0, fare ≤ 0?

**Hint:** one pass with `F.sum(F.col(c).isNull().cast("int"))` over `SRC_COLS` gives all null counts at once.

**Pause and observe →** Several columns show the *exact same* huge null count. Which columns, and which `payment_type` value has that same count? What does that tell you about those rows? Would deleting them be right? Decide, and defend it in report §6.

In [0]:
# Step 1: your code here

from pyspark.sql import functions as F

bronze_df = spark.table(f"{CATALOG}.bronze.taxi_trips")

# 1. How many rows in total, and how many nulls per column?
row_count = bronze_df.count()
null_counts = bronze_df.agg(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in SRC_COLS]
)
display(F.lit(row_count).alias("total_rows"))
display(null_counts)

# 2. Earliest and latest pickup; how many pickups outside source month?
pickup_minmax = bronze_df.agg(
    F.min("tpep_pickup_datetime").alias("earliest_pickup"),
    F.max("tpep_pickup_datetime").alias("latest_pickup")
)
pickups_wrong_month = bronze_df.withColumn(
    "pickup_month", F.date_format("tpep_pickup_datetime", "yyyy-MM")
).filter(F.col("pickup_month") != F.col("source_month")).count()
display(pickup_minmax)
display(F.lit(pickups_wrong_month).alias("pickups_outside_source_month"))

# 3. payment_type values and row counts
display(bronze_df.groupBy("payment_type").count().orderBy("payment_type"))

# 4. trips with dropoff ≤ pickup, distance ≤ 0, fare ≤ 0
invalid_cond = (
    (F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime")) |
    (F.col("trip_distance") <= 0) |
    (F.col("fare_amount") <= 0)
)
invalid_trips = bronze_df.filter(invalid_cond).count()
display(F.lit(invalid_trips).alias("invalid_trips"))

Column<'12253805 AS total_rows'>

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,0,0,2944316,0,2944316,2944316,0,0,0,0,0,0,0,0,0,0,2944316,2944316,0


earliest_pickup,latest_pickup
2009-01-01T12:52:15.000,2025-11-01T00:32:12.000


Column<'37 AS pickups_outside_source_month'>

payment_type,count
0,2944316
1,7777780
2,1141826
3,81837
4,308046


Column<'1290547 AS invalid_trips'>

---
## Step 2 — Split off the rejects

A row is **rejected** if it breaks any rule below. Each rejected row gets exactly **one** `reject_reason` (the **first** rule it fails, in this order):

| # | `reject_reason` (exact string) | Rejected when |
|---|---|---|
| 1 | `pickup_outside_source_month` | the pickup month ≠ `source_month` |
| 2 | `dropoff_not_after_pickup` | dropoff ≤ pickup |
| 3 | `duration_over_6h` | trip lasts more than 360 minutes |
| 4 | `distance_invalid` | `trip_distance` ≤ 0 or ≥ 100 miles |
| 5 | `fare_invalid` | `fare_amount` ≤ 0 or ≥ 500 |

Write rejected rows to `silver/taxi_trips_rejected/`: **all Bronze columns unchanged**, plus `reject_reason` (string) and `rejected_at` (timestamp).

**Hints**
- A chain of `F.when(...).when(...)` returns the first match, which is exactly "first failing rule".
- Keep the original column names and types here. This table is evidence for someone investigating.
- Count rejected rows per reason and keep **one example row per reason** for the report.

**Pause and observe →** Why does the table keep the *original* values instead of cleaned ones? Why one reason per row? Report §6.

In [0]:
# Step 2: your code here

bronze_df = spark.table(f"{CATALOG}.bronze.taxi_trips")

trip_duration_min = (
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

reject_reason = (
    F.when(F.date_format("tpep_pickup_datetime", "yyyy-MM") != F.col("source_month"), "pickup_outside_source_month")
     .when(F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"), "dropoff_not_after_pickup")
     .when(trip_duration_min > 360, "duration_over_6h")
     .when((F.col("trip_distance") <= 0) | (F.col("trip_distance") >= 100), "distance_invalid")
     .when((F.col("fare_amount") <= 0) | (F.col("fare_amount") >= 500), "fare_invalid")
)

rejected_df = bronze_df.withColumn("reject_reason", reject_reason) \
                       .filter(F.col("reject_reason").isNotNull()) \
                       .withColumn("rejected_at", F.current_timestamp())

# Write the rejected rows to Silver layer
spark.sql(
    f"ALTER TABLE {CATALOG}.silver.taxi_trips_rejected SET TBLPROPERTIES ('delta.feature.timestampNtz' = 'supported')"
)

rejected_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", f"{BASE}/silver/taxi_trips_rejected") \
    .saveAsTable(f"{CATALOG}.silver.taxi_trips_rejected")

# Show count per reject_reason and one example row per reason
display(
    rejected_df.groupBy("reject_reason").count().orderBy("reject_reason")
)
display(
    rejected_df.groupBy("reject_reason").agg(
        F.first(F.struct(*[F.col(c) for c in rejected_df.columns])).alias("example_row")
    ).select("reject_reason", "example_row")
)

reject_reason,count
distance_invalid,351499
dropoff_not_after_pickup,172980
duration_over_6h,3490
fare_invalid,766477
pickup_outside_source_month,37


reject_reason,example_row
distance_invalid,"List(2, 2025-10-01T00:12:43.000, 2025-10-01T00:25:05.000, 1, 0.0, 1, N, 186, 186, 1, 11.4, 1.0, 0.5, 2.78, 0.0, 1.0, 16.68, 0.0, 0.0, 0.0, yellow_tripdata_2025-10.parquet, 2025-10, 2026-11-12T16:15:45.345Z, distance_invalid, 2026-11-12T21:13:54.407Z)"
dropoff_not_after_pickup,"List(7, 2025-10-01T00:00:08.000, 2025-10-01T00:00:08.000, 1, 5.0, 1, N, 107, 225, 1, 28.2, 0.0, 0.5, 8.49, 0.0, 1.0, 42.44, 2.5, 0.0, 0.75, yellow_tripdata_2025-10.parquet, 2025-10, 2026-11-12T16:15:45.345Z, dropoff_not_after_pickup, 2026-11-12T21:13:54.407Z)"
pickup_outside_source_month,"List(2, 2025-09-30T23:59:51.000, 2025-10-01T00:09:07.000, 1, 2.0, 1, N, 230, 158, 1, 11.4, 1.0, 0.5, 3.43, 0.0, 1.0, 20.58, 2.5, 0.0, 0.75, yellow_tripdata_2025-10.parquet, 2025-10, 2026-11-12T16:15:45.345Z, pickup_outside_source_month, 2026-11-12T21:13:54.407Z)"
duration_over_6h,"List(2, 2025-10-01T00:06:43.000, 2025-10-01T23:21:42.000, 1, 2.7, 1, N, 170, 143, 1, 14.9, 1.0, 0.5, 2.06, 0.0, 1.0, 22.71, 2.5, 0.0, 0.75, yellow_tripdata_2025-10.parquet, 2025-10, 2026-11-12T16:15:45.345Z, duration_over_6h, 2026-11-12T21:13:54.407Z)"
fare_invalid,"List(2, 2025-10-01T00:49:47.000, 2025-10-01T00:58:09.000, 1, 2.29, 1, N, 142, 246, 4, -12.1, -1.0, -0.5, 0.0, 0.0, -1.0, -17.85, -2.5, 0.0, -0.75, yellow_tripdata_2025-10.parquet, 2025-10, 2026-11-12T16:15:45.345Z, fare_invalid, 2026-11-12T21:13:54.407Z)"


---
## Step 3 — Clean the good rows into `silver.taxi_trips`

Take the rows that passed Step 2, **remove exact duplicates** (all 20 source columns equal; count how many you removed into a variable named `DUPLICATES_REMOVED`), and shape them exactly like this:

| Bronze column | Silver column | Silver type | Note |
|---|---|---|---|
| `VendorID` | `vendor_id` | int | |
| `tpep_pickup_datetime` | `pickup_ts` | timestamp | |
| `tpep_dropoff_datetime` | `dropoff_ts` | timestamp | |
| `passenger_count` | `passenger_count` | int | outside 1–6 → `NULL` (keep the trip, fix the value) |
| `trip_distance` | `trip_distance_miles` | decimal(8,2) | |
| `RatecodeID` | `rate_code_id` | int | |
| `store_and_fwd_flag` | `store_and_fwd_flag` | boolean | `Y` → true, `N` → false, anything else → null |
| `PULocationID` | `pickup_location_id` | int | |
| `DOLocationID` | `dropoff_location_id` | int | |
| `payment_type` | `payment_type_id` | int | |
| *(new)* | `payment_type_desc` | string | 0 Flex fare · 1 Credit card · 2 Cash · 3 No charge · 4 Dispute · 5 Unknown · 6 Voided trip · else `Other` |
| `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `cbd_congestion_fee` | same names | decimal(10,2) | money is never a float |
| `Airport_fee` | `airport_fee` | decimal(10,2) | snake_case |
| *(new)* | `pickup_date` | date | from the pickup timestamp |
| *(new)* | `pickup_hour` | int | 0–23 |
| *(new)* | `trip_duration_min` | decimal(8,2) | (dropoff − pickup) in minutes |
| `source_month` | `source_month` | string | carried over |
| *(new)* | `silver_loaded_at` | timestamp | |

Columns not in this table (`source_file`, `ingested_at`) are **not** carried to Silver. The column order above is the order to use.

**Hints**
- `dropDuplicates(SRC_COLS)` removes exact duplicates; the difference of two counts tells you how many.
- `.cast("decimal(10,2)")` for money. An unknown payment type must become `Other`, never an error.
- `F.to_date`, `F.hour`, `F.unix_timestamp` cover the derived columns.
- Write with `overwrite` to `silver/taxi_trips/`, then `printSchema()` and count per `source_month`.

**Pause and observe →** Why decimal for money? Why set an out-of-range passenger count to `NULL` rather than reject the trip, when you *rejected* a ≤ 0 fare? Report §7 asks you to fill a before/after schema table.

In [0]:
# Step 3: your code here

# Select only good rows (not rejected)
bronze_df = spark.table(f"{CATALOG}.bronze.taxi_trips")
rejected_df = spark.table(f"{CATALOG}.silver.taxi_trips_rejected")

join_condition = None
for c in SRC_COLS:
    predicate = F.col(f"b.{c}").eqNullSafe(F.col(f"r.{c}"))
    join_condition = predicate if join_condition is None else (join_condition & predicate)

good_rows_df = bronze_df.alias("b").join(
    rejected_df.select(*SRC_COLS).alias("r"), join_condition, "left_anti"
)

# Remove exact duplicates
silver_count_before_dedup = good_rows_df.count()
deduped_df = good_rows_df.dropDuplicates(SRC_COLS)
DUPLICATES_REMOVED = silver_count_before_dedup - deduped_df.count()

# Add and transform columns in specified order
payment_type_desc_map = {
    0: "Flex fare", 1: "Credit card", 2: "Cash", 3: "No charge",
    4: "Dispute", 5: "Unknown", 6: "Voided trip"
}

silver_df = deduped_df.select(
    F.col("VendorID").cast("int").alias("vendor_id"),
    F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_ts"),
    F.col("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_ts"),
    F.when((F.col("passenger_count") < 1) | (F.col("passenger_count") > 6), None)
     .otherwise(F.col("passenger_count").cast("int")).alias("passenger_count"),
    F.col("trip_distance").cast("decimal(8,2)").alias("trip_distance_miles"),
    F.col("RatecodeID").cast("int").alias("rate_code_id"),
    F.when(F.col("store_and_fwd_flag") == "Y", True)
     .when(F.col("store_and_fwd_flag") == "N", False)
     .otherwise(None).alias("store_and_fwd_flag"),
    F.col("PULocationID").cast("int").alias("pickup_location_id"),
    F.col("DOLocationID").cast("int").alias("dropoff_location_id"),
    F.col("payment_type").cast("int").alias("payment_type_id"),
    F.when(
        F.col("payment_type").cast("int").isin(list(payment_type_desc_map.keys())),
        F.expr(f"CASE payment_type \
            WHEN 0 THEN 'Flex fare' WHEN 1 THEN 'Credit card' WHEN 2 THEN 'Cash' \
            WHEN 3 THEN 'No charge' WHEN 4 THEN 'Dispute' WHEN 5 THEN 'Unknown' \
            WHEN 6 THEN 'Voided trip' ELSE 'Other' END")
    ).otherwise("Other").alias("payment_type_desc"),
    F.col("fare_amount").cast("decimal(10,2)").alias("fare_amount"),
    F.col("extra").cast("decimal(10,2)").alias("extra"),
    F.col("mta_tax").cast("decimal(10,2)").alias("mta_tax"),
    F.col("tip_amount").cast("decimal(10,2)").alias("tip_amount"),
    F.col("tolls_amount").cast("decimal(10,2)").alias("tolls_amount"),
    F.col("improvement_surcharge").cast("decimal(10,2)").alias("improvement_surcharge"),
    F.col("total_amount").cast("decimal(10,2)").alias("total_amount"),
    F.col("congestion_surcharge").cast("decimal(10,2)").alias("congestion_surcharge"),
    F.col("Airport_fee").cast("decimal(10,2)").alias("airport_fee"),
    F.col("cbd_congestion_fee").cast("decimal(10,2)").alias("cbd_congestion_fee"),
    F.to_date("tpep_pickup_datetime").alias("pickup_date"),
    F.hour("tpep_pickup_datetime").alias("pickup_hour"),
    ((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
        .cast("decimal(8,2)").alias("trip_duration_min"),
    F.col("source_month"),
    F.current_timestamp().alias("silver_loaded_at")
)

silver_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", f"{BASE}/silver/taxi_trips") \
    .saveAsTable(f"{CATALOG}.silver.taxi_trips")

silver_df.printSchema()
display(
    silver_df.groupBy("source_month").count().orderBy("source_month")
)

root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)
 |-- dropoff_ts: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance_miles: decimal(8,2) (nullable = true)
 |-- rate_code_id: integer (nullable = true)
 |-- store_and_fwd_flag: boolean (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- payment_type_id: integer (nullable = true)
 |-- payment_type_desc: string (nullable = false)
 |-- fare_amount: decimal(10,2) (nullable = true)
 |-- extra: decimal(10,2) (nullable = true)
 |-- mta_tax: decimal(10,2) (nullable = true)
 |-- tip_amount: decimal(10,2) (nullable = true)
 |-- tolls_amount: decimal(10,2) (nullable = true)
 |-- improvement_surcharge: decimal(10,2) (nullable = true)
 |-- total_amount: decimal(10,2) (nullable = true)
 |-- congestion_surcharge: decimal(10,2) (nullable = true)
 |-- airport_fee: decimal(10,2) (nullable = true)
 |

source_month,count
2025-08,3179661
2025-09,3837040
2025-10,3942621


---
## Step 4 — Silver weather

Read `bronze.weather_daily` and write `silver/weather_daily/`:

| Bronze column | Silver column | Silver type | Note |
|---|---|---|---|
| `time` | `weather_date` | date | one row per date, drop duplicates on the date |
| `temperature_2m_max` | `temp_max_c` | decimal(5,1) | |
| `temperature_2m_min` | `temp_min_c` | decimal(5,1) | |
| `precipitation_sum` | `precip_mm` | decimal(6,1) | |
| `snowfall_sum` | `snowfall_cm` | decimal(6,1) | |
| `wind_speed_10m_max` | `wind_max_kmh` | decimal(6,1) | |
| *(new)* | `is_rainy` | boolean | true when precipitation is **1.0 mm or more** |
| *(new)* | `silver_loaded_at` | timestamp | |

**Pause and observe →** Why encode the units (`_c`, `_mm`, `_cm`, `_kmh`) in the column names? Report §7.

In [0]:
# Step 4: your code here

weather_df = spark.table(f"{CATALOG}.bronze.weather_daily")

silver_weather_df = (
    weather_df
    .dropDuplicates(["time"])
    .select(
        F.col("time").cast("date").alias("weather_date"),
        F.col("temperature_2m_max").cast("decimal(5,1)").alias("temp_max_c"),
        F.col("temperature_2m_min").cast("decimal(5,1)").alias("temp_min_c"),
        F.col("precipitation_sum").cast("decimal(6,1)").alias("precip_mm"),
        F.col("snowfall_sum").cast("decimal(6,1)").alias("snowfall_cm"),
        F.col("wind_speed_10m_max").cast("decimal(6,1)").alias("wind_max_kmh"),
        (F.col("precipitation_sum") >= 1.0).alias("is_rainy"),
        F.current_timestamp().alias("silver_loaded_at")
    )
)

silver_weather_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", f"{BASE}/silver/weather_daily") \
    .saveAsTable(f"{CATALOG}.silver.weather_daily")

silver_weather_df.printSchema()
display(silver_weather_df.orderBy("weather_date"))


root
 |-- weather_date: date (nullable = true)
 |-- temp_max_c: decimal(5,1) (nullable = true)
 |-- temp_min_c: decimal(5,1) (nullable = true)
 |-- precip_mm: decimal(6,1) (nullable = true)
 |-- snowfall_cm: decimal(6,1) (nullable = true)
 |-- wind_max_kmh: decimal(6,1) (nullable = true)
 |-- is_rainy: boolean (nullable = true)
 |-- silver_loaded_at: timestamp (nullable = false)



weather_date,temp_max_c,temp_min_c,precip_mm,snowfall_cm,wind_max_kmh,is_rainy,silver_loaded_at
2025-08-01,23.0,17.8,1.0,0.0,16.1,true,2026-11-12T21:20:40.317Z
2025-08-02,25.7,17.1,0.0,0.0,10.1,false,2026-11-12T21:20:40.317Z
2025-08-03,28.1,16.5,0.0,0.0,11.1,false,2026-11-12T21:20:40.317Z
2025-08-04,33.2,17.0,0.0,0.0,14.6,false,2026-11-12T21:20:40.317Z
2025-08-05,31.0,19.9,0.0,0.0,9.7,false,2026-11-12T21:20:40.317Z
2025-08-06,28.5,20.3,0.9,0.0,13.0,false,2026-11-12T21:20:40.317Z
2025-08-07,28.6,19.0,0.0,0.0,13.8,false,2026-11-12T21:20:40.317Z
2025-08-08,28.2,17.0,0.0,0.0,12.8,false,2026-11-12T21:20:40.317Z
2025-08-09,30.0,17.3,0.0,0.0,11.3,false,2026-11-12T21:20:40.317Z
2025-08-10,33.3,18.3,0.0,0.0,13.0,false,2026-11-12T21:20:40.317Z


---
## Step 5 — Register, reconcile and sanity-check

1. Create schema `silver` and register the three **external** tables from the contract table.
2. Run the check cell below (given). Fix every ❌. It also reconciles **bronze = silver + rejected + duplicates removed**.
3. Write **one query** of your own that joins `silver.taxi_trips` to `silver.weather_daily` on the date and returns **trips per day for rainy vs dry days**. Put the result in report §9. This is only a sanity check that the two tables fit together, not an analysis.

> **Git checkpoint:** commit this notebook and your filled report (`git commit -m "notebook 2: silver layer + report"`).

In [0]:
# Step 5: register the three external tables here

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.taxi_trips
USING DELTA
LOCATION '{BASE}/silver/taxi_trips'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.taxi_trips_rejected
USING DELTA
LOCATION '{BASE}/silver/taxi_trips_rejected'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.weather_daily
USING DELTA
LOCATION '{BASE}/silver/weather_daily'
""")


DataFrame[]

In [0]:
# Step 5: your rainy-vs-dry join query here

display(
    spark.table(f"{CATALOG}.silver.taxi_trips")
    .join(
        spark.table(f"{CATALOG}.silver.weather_daily"),
        F.col("pickup_date") == F.col("weather_date"),
        "inner"
    )
    .groupBy("pickup_date", "is_rainy")
    .count()
    .withColumn(
        "day_type",
        F.when(F.col("is_rainy"), F.lit("rainy")).otherwise(F.lit("dry"))
    )
    .select("pickup_date", "day_type", F.col("count").alias("trips_per_day"))
    .orderBy("pickup_date", "day_type")
)


pickup_date,day_type,trips_per_day
2025-08-01,rainy,118038
2025-08-02,dry,123058
2025-08-03,dry,106067
2025-08-04,dry,100444
2025-08-05,dry,112490
2025-08-06,dry,121400
2025-08-07,dry,116479
2025-08-08,dry,110323
2025-08-09,dry,109950
2025-08-10,dry,98020


In [0]:


from pyspark.sql import functions as F

_results = []
def check(label, ok):
    _results.append(bool(ok))
    print(("✅" if ok else "❌"), label)

def table_info(name):
    return {r.col_name: r.data_type for r in spark.sql(f"DESCRIBE EXTENDED {name}").collect()}

def summary():
    print(f"\n{sum(_results)} of {len(_results)} checks passed")

EXPECTED_SILVER_TAXI = {
    "vendor_id": "int", "pickup_ts": "timestamp", "dropoff_ts": "timestamp", "passenger_count": "int",
    "trip_distance_miles": "decimal(8,2)", "rate_code_id": "int", "store_and_fwd_flag": "boolean",
    "pickup_location_id": "int", "dropoff_location_id": "int", "payment_type_id": "int",
    "payment_type_desc": "string", "fare_amount": "decimal(10,2)", "extra": "decimal(10,2)",
    "mta_tax": "decimal(10,2)", "tip_amount": "decimal(10,2)", "tolls_amount": "decimal(10,2)",
    "improvement_surcharge": "decimal(10,2)", "total_amount": "decimal(10,2)",
    "congestion_surcharge": "decimal(10,2)", "airport_fee": "decimal(10,2)", "cbd_congestion_fee": "decimal(10,2)",
    "pickup_date": "date", "pickup_hour": "int", "trip_duration_min": "decimal(8,2)",
    "source_month": "string", "silver_loaded_at": "timestamp",
}
EXPECTED_SILVER_WEATHER = {
    "weather_date": "date", "temp_max_c": "decimal(5,1)", "temp_min_c": "decimal(5,1)",
    "precip_mm": "decimal(6,1)", "snowfall_cm": "decimal(6,1)", "wind_max_kmh": "decimal(6,1)",
    "is_rainy": "boolean", "silver_loaded_at": "timestamp",
}
ALLOWED_REASONS = {"pickup_outside_source_month", "dropoff_not_after_pickup", "duration_over_6h",
                   "distance_invalid", "fare_invalid"}

# ---------------- external tables in the right place ----------------
for t in ["taxi_trips", "taxi_trips_rejected", "weather_daily"]:
    info = table_info(f"{CATALOG}.silver.{t}")
    check(f"silver.{t} is EXTERNAL at <container>/silver/{t}",
          info.get("Type") == "EXTERNAL" and info.get("Location", "").rstrip("/").endswith(f"/silver/{t}"))

# ---------------- silver taxi: contract ----------------
st = spark.table(f"{CATALOG}.silver.taxi_trips")
check("silver.taxi_trips has exactly the agreed columns and types", dict(st.dtypes) == EXPECTED_SILVER_TAXI)
stats = st.agg(
    F.min("trip_duration_min").alias("min_dur"), F.max("trip_duration_min").alias("max_dur"),
    F.min("fare_amount").alias("min_fare"), F.max("fare_amount").alias("max_fare"),
    F.min("trip_distance_miles").alias("min_dist"), F.max("trip_distance_miles").alias("max_dist"),
    F.min("passenger_count").alias("min_pax"), F.max("passenger_count").alias("max_pax"),
    F.sum((F.date_format("pickup_ts", "yyyy-MM") != F.col("source_month")).cast("int")).alias("wrong_month"),
).first()
check("no trip with duration <= 0 or > 360 min", stats.min_dur > 0 and stats.max_dur <= 360)
check("no fare <= 0 or >= 500", stats.min_fare > 0 and stats.max_fare < 500)
check("no distance <= 0 or >= 100", stats.min_dist > 0 and stats.max_dist < 100)
check("passenger_count is NULL or between 1 and 6", stats.min_pax >= 1 and stats.max_pax <= 6)
check("every pickup falls in its own source_month", stats.wrong_month == 0)

# ---------------- rejected table ----------------
rj = spark.table(f"{CATALOG}.silver.taxi_trips_rejected")
check("rejected table has reject_reason and rejected_at", {"reject_reason", "rejected_at"} <= set(rj.columns))
reasons = {r.reject_reason for r in rj.select("reject_reason").distinct().collect()}
check("every reject_reason is one of the 5 agreed strings", reasons <= ALLOWED_REASONS)

# ---------------- reconciliation: nothing disappears silently ----------------
b_n, s_n, r_n = spark.table(f"{CATALOG}.bronze.taxi_trips").count(), st.count(), rj.count()
check(f"bronze ({b_n:,}) == silver ({s_n:,}) + rejected ({r_n:,}) + duplicates removed ({DUPLICATES_REMOVED:,})",
      b_n == s_n + r_n + DUPLICATES_REMOVED)

# ---------------- silver weather ----------------
sw = spark.table(f"{CATALOG}.silver.weather_daily")
check("silver.weather_daily has the agreed columns and types", dict(sw.dtypes) == EXPECTED_SILVER_WEATHER)
check("weather: 92 rows, one per date", sw.count() == 92 and sw.select("weather_date").distinct().count() == 92)

summary()
print(f"Bronze {b_n:,} | Silver {s_n:,} | Rejected {r_n:,} | Duplicates removed {DUPLICATES_REMOVED:,}")

✅ silver.taxi_trips is EXTERNAL at <container>/silver/taxi_trips
✅ silver.taxi_trips_rejected is EXTERNAL at <container>/silver/taxi_trips_rejected
✅ silver.weather_daily is EXTERNAL at <container>/silver/weather_daily
✅ silver.taxi_trips has exactly the agreed columns and types
✅ no trip with duration <= 0 or > 360 min
✅ no fare <= 0 or >= 500
✅ no distance <= 0 or >= 100
✅ passenger_count is NULL or between 1 and 6
✅ every pickup falls in its own source_month
✅ rejected table has reject_reason and rejected_at
✅ every reject_reason is one of the 5 agreed strings
✅ bronze (12,253,805) == silver (10,959,322) + rejected (1,294,483) + duplicates removed (0)
✅ silver.weather_daily has the agreed columns and types
✅ weather: 92 rows, one per date

14 of 14 checks passed
Bronze 12,253,805 | Silver 10,959,322 | Rejected 1,294,483 | Duplicates removed 0
